# Day 57 · Human in the loop: where the approval step belongs

**First-principles question:** Which decisions should a person make, and what does that cost?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import pandas as pd
from lab.acme import INTENTS, load_cases, load_results
from lab.evals import score

cases = {c["id"]: c for c in load_cases()}
VERSION = "C"          # the version you are recommending (A, B, B_retrieved or C)
rows = load_results(VERSION)

## 1 · What does each kind of error cost Acme?
Your estimates in ₹ (a refund paid wrongly, a broken promise that brings a complaint, a missed fraud report, a rude reply that loses a customer…).

In [ ]:
# TODO: your estimates
ERROR_COST = {
    "wrong facts or policy": None,      # e.g. a promise that leads to a complaint and a credit
    "missed escalation": None,          # needs_human should have been true
    "wrong intent": None,
}
HUMAN_COST_PER_EMAIL = None             # an agent's time for one email, in ₹

## 2 · Error rates per intent

In [ ]:
records = []
for r in rows:
    case, s = cases[r["id"]], score(cases[r["id"]], r)
    records.append({"intent": case["expected_intent"],
                    "wrong facts or policy": not (s["facts_ok"] and s["policy_ok"]),
                    "missed escalation": case["needs_human"] and not r.get("needs_human"),
                    "wrong intent": not s["intent_ok"]})
rates = pd.DataFrame(records).groupby("intent").mean()
rates

## 3 · Three policies, cost per 1,000 emails

In [ ]:
share = pd.Series([cases[r["id"]]["expected_intent"] for r in rows]).value_counts(normalize=True)
auto_cost = (rates * pd.Series(ERROR_COST)).sum(axis=1)       # expected error cost per email, by intent

# TODO: RISKY = the intents you would always send to a human
RISKY = []

def cost_per_1000(to_human):
    total = 0.0
    for intent, p in share.items():
        total += p * (HUMAN_COST_PER_EMAIL if intent in to_human else auto_cost.get(intent, 0))
    return 1000 * total

policies = {"all automatic": [], f"risky to humans {RISKY}": RISKY, "all humans": list(share.index)}
pd.Series({name: cost_per_1000(h) for name, h in policies.items()}, name="₹ per 1,000 emails").round(0)

**My policy, and why:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert None not in ERROR_COST.values() and HUMAN_COST_PER_EMAIL is not None, "Step 1: fill in the costs"
assert RISKY, "Step 3: choose at least one risky intent"
assert all(i in INTENTS for i in RISKY)
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")